# 🌍 Polyglot Live — Google Colab Host & ngrok Tunneling

This notebook automates setting up **Polyglot Live** in a Google Colab GPU instance.
It compiles and configures:
1. **llama-server** (llama.cpp) running GPU offloading for a multimodal/vision Gemma 4 model.
2. **Supertonic V3 TTS** running on ONNX GPU runtime for high-fidelity speech synthesis.
3. **Polyglot Live Web UI** exposed securely to your browser via an `ngrok` tunnel (with dynamic `wss://` secure WebSocket support).

### Instructions:
1. Go to **Runtime > Change runtime type** and ensure you have selected a **GPU** accelerator (e.g. T4, L4, or A100).
2. Run the cells sequentially.

## 🔍 Step 1: Verify GPU & Environment

Check that Google Colab has successfully allocated a GPU runtime and check the CUDA environment version.

In [ ]:
# Verify GPU availability & CUDA version
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

result2 = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
print(result2.stdout)

## 🛠️ Step 2: Install System Prerequisites

We need to install the compiler tools and libraries required to build the WebRTC AEC (`aec-audio-processing`) package from source, along with audio handling libraries like PortAudio.

In [ ]:
!apt-get update && apt-get install -y swig build-essential meson libsndfile1 libportaudio2 portaudio19-dev psmisc

## 📂 Step 3: Clone the Polyglot Repository

Clone the latest repository code from GitHub and enter the directory.

In [ ]:
# Clean up any previous attempts to prevent nested directory cloning
%cd /content
!rm -rf /content/polyglot
!git clone https://github.com/kishore-1404/polyglot.git
%cd /content/polyglot

## 📦 Step 4: Install Python Dependencies & ONNX Runtime GPU

We install the required Python packages. We uninstall standard `onnxruntime` first to avoid conflicts with `onnxruntime-gpu` (which enables GPU acceleration for Supertonic TTS), then install the requirements and `pyngrok` for tunneling.

In [ ]:
# Uninstall CPU version of ORT first to avoid conflict with GPU version
!pip uninstall -y onnxruntime
# Install project requirements
!pip install -r requirements.txt
# Install GPU version of ONNX Runtime from Microsoft's CUDA 12 nightly feed, and pyngrok
!pip install onnxruntime-gpu --extra-index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/ pyngrok -q

## 🏎️ Step 5: Fetch and Install llama.cpp with CUDA Support

We query the latest `llama.cpp` CUDA-optimized binary release matching the machine's architecture and extract it.

In [ ]:
import urllib.request, json, platform, tarfile, os

api_url = 'https://api.github.com/repos/ai-dock/llama.cpp-cuda/releases/latest'
with urllib.request.urlopen(api_url) as response:
    data = json.loads(response.read().decode())

latest_tag   = data['tag_name']
release_name = data['name']
published_at = data['published_at']

print(f'Latest release tag : {latest_tag}')
print(f'Release name       : {release_name}')

machine = platform.machine().lower()
arch = 'amd64' if machine in ('x86_64', 'amd64') else 'arm64'
print(f'Host architecture  : {machine} → \'{arch}\' build')

CUDA_VERSION = '12.8'
target_suffix = f'cuda-{CUDA_VERSION}-{arch}.tar.gz'
selected_asset = None

for asset in data['assets']:
    if asset['name'].endswith(target_suffix):
        selected_asset = asset
        break

if selected_asset is None:
    # Fallback to any cuda tarball
    for asset in data['assets']:
        if 'cuda' in asset['name'] and asset['name'].endswith('.tar.gz'):
            selected_asset = asset
            break

if selected_asset is None:
    raise RuntimeError('No suitable CUDA asset found in the release!')

TARBALL = selected_asset['name']
DOWNLOAD_URL = selected_asset['browser_download_url']
TARBALL_PATH = f'/tmp/{TARBALL}'
INSTALL_DIR = '/opt/llama.cpp'

print(f'Downloading {TARBALL}...')
def reporthook(count, block_size, total_size):
    pct = count * block_size * 100 // total_size
    if count % 100 == 0:
        print(f'  {pct}% ({count * block_size // (1024*1024)} MB / {total_size // (1024*1024)} MB)')

urllib.request.urlretrieve(DOWNLOAD_URL, TARBALL_PATH, reporthook=reporthook)
print(f'✅ Download complete: {TARBALL_PATH}')

os.makedirs(INSTALL_DIR, exist_ok=True)
print(f'Extracting to {INSTALL_DIR}...')
with tarfile.open(TARBALL_PATH, 'r:gz') as tar:
    tar.extractall(INSTALL_DIR)
print('✅ Extraction complete')

# Locate the extracted directory
extracted_dirs = [d for d in os.listdir(INSTALL_DIR) if os.path.isdir(os.path.join(INSTALL_DIR, d))]
if extracted_dirs:
    bin_sub = extracted_dirs[0]
    BIN_DIR = os.path.join(INSTALL_DIR, bin_sub)
else:
    BIN_DIR = INSTALL_DIR
print(f'Llama binary directory: {BIN_DIR}')

## 📥 Step 6: Download Gemma 4 Model & multimodal mmproj GGUF

Download the core Gemma 4 model (`gemma-4-E4B-it-qat-UD-Q4_K_XL.gguf`) and the specified multimodal projector (`mmproj-F16.gguf`) for vision features.

In [ ]:
import os, urllib.request

MODEL_DIR = "/content/models"
os.makedirs(MODEL_DIR, exist_ok=True)

models_to_download = {
    "gemma-4-E4B-it-qat-UD-Q4_K_XL.gguf": "https://huggingface.co/unsloth/gemma-4-E4B-it-qat-GGUF/resolve/main/gemma-4-E4B-it-qat-UD-Q4_K_XL.gguf",
    "mmproj-F16.gguf": "https://huggingface.co/unsloth/gemma-4-E4B-it-GGUF/resolve/main/mmproj-F16.gguf"
}

for name, url in models_to_download.items():
    path = os.path.join(MODEL_DIR, name)
    if os.path.exists(path):
        print(f"✅ Model already exists: {path} ({os.path.getsize(path) / (1024**3):.2f} GB)")
    else:
        print(f"Downloading {name}...")
        downloaded = [0]
        def progress(count, block_size, total_size):
            downloaded[0] = count * block_size
            pct = min(downloaded[0] * 100 // total_size, 100)
            done = pct // 5
            bar = "█" * done + "░" * (20 - done)
            print(f"\r  [{bar}] {pct}%  ({downloaded[0]/(1024**3):.2f} / {total_size/(1024**3):.2f} GB)", end="")
        urllib.request.urlretrieve(url, path, reporthook=progress)
        print(f"\n✅ Download complete: {path}\n")

## ⚙️ Step 7: Configure config.yaml

Dynamically update paths in `config.yaml` to point to the downloaded models and configure the local host ports.

In [ ]:
import yaml

with open("config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

cfg["models"]["llm_gguf"] = "/content/models/gemma-4-E4B-it-qat-UD-Q4_K_XL.gguf"
cfg["models"]["mmproj_gguf"] = "/content/models/mmproj-F16.gguf"
cfg["llama_server"]["port"] = 8088
cfg["supertonic"]["port"] = 9988
cfg["ui"]["web_port"] = 7860

with open("config.yaml", "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

print("✅ config.yaml updated:")
print(yaml.dump(cfg, default_flow_style=False))

## 🚀 Step 8: Launch Backend Servers in the Background

We construct the correct environment paths to point to CUDA libraries so both `llama-server` and `onnxruntime-gpu` (used by Supertonic) run with full GPU acceleration. We launch both processes in the background and monitor their status.

In [ ]:
import sys, os, subprocess, time, threading, requests

# Fallback logic for BIN_DIR if not defined in local namespace
if 'BIN_DIR' not in globals():
    INSTALL_DIR = '/opt/llama.cpp'
    extracted_dirs = [d for d in os.listdir(INSTALL_DIR) if os.path.isdir(os.path.join(INSTALL_DIR, d))]
    if extracted_dirs:
        BIN_DIR = os.path.join(INSTALL_DIR, extracted_dirs[0])
    else:
        BIN_DIR = INSTALL_DIR

# Find python packages directory to locate PyPI nvidia libraries
site_packages = [p for p in sys.path if 'site-packages' in p or 'dist-packages' in p]
nvidia_libs = []
for sp in site_packages:
    nvidia_dir = os.path.join(sp, 'nvidia')
    if os.path.isdir(nvidia_dir):
        for sub in ['cublas/lib', 'cufft/lib', 'cuda_runtime/lib', 'cuda_nvrtc/lib', 'nvjitlink/lib']:
            p = os.path.join(nvidia_dir, sub)
            if os.path.isdir(p):
                nvidia_libs.append(p)

# Kill any existing llama-server, supertonic or web processes explicitly
subprocess.run(['pkill', '-f', 'llama-server'], capture_output=True)
subprocess.run(['pkill', '-f', 'supertonic'], capture_output=True)
for port in [8088, 9988, 7860]:
    subprocess.run(['fuser', '-k', f'{port}/tcp'], capture_output=True)
time.sleep(2)

# Set up library paths, linking to PyPI nvidia libraries and compiled llama.cpp
os.environ['LD_LIBRARY_PATH'] = ':'.join(nvidia_libs) + ':' + BIN_DIR + ':' + '/usr/lib64-nvidia' + ':/usr/lib/x86_64-linux-gnu:/usr/lib:/lib:' + os.environ.get('LD_LIBRARY_PATH', '')
os.environ['PATH'] = BIN_DIR + ':' + os.environ.get('PATH', '')

print(f'LD_LIBRARY_PATH: {os.environ["LD_LIBRARY_PATH"]}')

# 1. Start llama-server
llama_cmd = [
    'llama-server',
    '-m', '/content/models/gemma-4-E4B-it-qat-UD-Q4_K_XL.gguf',
    '--mmproj', '/content/models/mmproj-F16.gguf',
    '--host', '0.0.0.0',
    '--port', '8088',
    '-ngl', '999',
    '-c', '32768',
    '--flash-attn', 'on',
    '--no-mmap'
]

print('🚀 Starting llama-server...')
llama_proc = subprocess.Popen(
    llama_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=os.environ
)

# 2. Start supertonic serve
tts_cmd = [
    'supertonic', 'serve',
    '--port', '9988'
]

print('🚀 Starting Supertonic V3 serve...')
tts_proc = subprocess.Popen(
    tts_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=os.environ
)

# Monitoring threads
llama_ready = threading.Event()
tts_ready = threading.Event()

def monitor_llama():
    for line in llama_proc.stdout:
        print(f'[llama-server] {line.strip()}')
        if 'all slots are idle' in line.lower() or 'HTTP server listening' in line:
            llama_ready.set()

def monitor_tts():
    for line in tts_proc.stdout:
        print(f'[supertonic] {line.strip()}')
        if 'Uvicorn running' in line or 'Application startup complete' in line:
            tts_ready.set()

t1 = threading.Thread(target=monitor_llama, daemon=True)
t2 = threading.Thread(target=monitor_tts, daemon=True)
t1.start()
t2.start()

# Wait for servers
print('⏳ Waiting for servers to warm up...')
timeout = 180
start_time = time.time()
while time.time() - start_time < timeout:
    if llama_ready.is_set() and tts_ready.is_set():
        break
    # Double check via API endpoints
    if not llama_ready.is_set():
        try:
            if requests.get('http://localhost:8088/health', timeout=1).ok:
                llama_ready.set()
                print('✅ llama-server responded to health check')
        except Exception:
            pass
    if not tts_ready.is_set():
        try:
            r = requests.post('http://localhost:9988/v1/tts', json={'text': 'ok', 'lang': 'en', 'voice': 'F1'}, timeout=1)
            if r.ok:
                tts_ready.set()
                print('✅ Supertonic TTS responded to health check')
            else:
                print(f'⚠️ Supertonic responded with status {r.status_code}: {r.text[:200]}')
        except Exception as e:
            pass
    time.sleep(3)

if llama_ready.is_set() and tts_ready.is_set():
    print('\n🎉 Both servers are online and ready!')
else:
    print('\n⚠️ Startup timed out. Check logs above.')

## 🌐 Step 9: Launch the Web UI

Launch the primary Polyglot web application in the background (binding to port 7860).

In [ ]:
# Kill any process on web port
import sys, os, subprocess, time, threading, requests
subprocess.run(['pkill', '-f', 'main.py'], capture_output=True)
subprocess.run(['fuser', '-k', '7860/tcp'], capture_output=True)
time.sleep(1)

# Ensure we are in the correct root directory and paths are loaded
%cd /content/polyglot
os.environ['LD_LIBRARY_PATH'] = '/usr/lib/x86_64-linux-gnu:/usr/lib:/lib:' + os.environ.get('LD_LIBRARY_PATH', '')

print("🚀 Starting Polyglot Web Application...")
web_proc = subprocess.Popen(
    [sys.executable, "-u", "main.py", "--web"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=os.environ
)

web_ready = threading.Event()
def monitor_web():
    for line in web_proc.stdout:
        print(f"[web-ui] {line.strip()}")
        if "Web UI →" in line or "Pipeline running" in line:
            web_ready.set()

t3 = threading.Thread(target=monitor_web, daemon=True)
t3.start()

# Wait for Web UI to be ready
start_time = time.time()
while time.time() - start_time < 30:
    if web_ready.is_set():
        break
    try:
        if requests.get("http://localhost:7860/", timeout=1).ok:
            web_ready.set()
            break
    except Exception:
        pass
    time.sleep(1)

if web_ready.is_set():
    print("✅ Web Application started on port 7860!")
else:
    print("⚠️ Web Application did not start in time. Check logs above.")

## 🔗 Step 10: Expose Web UI via ngrok Tunnel

To access the Web UI from your browser, enter your ngrok Auth Token below. 
You can get your token for free at: [ngrok Dashboard](https://dashboard.ngrok.com/get-started/your-authtoken)

In [ ]:
import getpass
from pyngrok import ngrok

# Ask for authtoken
NGROK_TOKEN = getpass.getpass("Enter your ngrok Authtoken: ")
ngrok.set_auth_token(NGROK_TOKEN)

# Disconnect existing tunnels
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)
    print(f"Closed old tunnel: {t.public_url}")

# Open secure HTTP tunnel on port 7860
tunnel = ngrok.connect("127.0.0.1:7860", "http")
PUBLIC_URL = tunnel.public_url

print("\n" + "="*60)
print(f"🎉 Tunnel active!")
print(f"👉 Access your Polyglot companion at: {PUBLIC_URL}")
print("="*60)
print("\nUse the control panel in your browser to run automated Demo Mode scenarios.")